# 05 - County-level flow-change maps (100-mi)  -  Helene x Milton

2 rows (storms) x 3 cols (within / inflow / outflow). Metric per column: within->`largest_drop`,
inflow->`largest_drop`, outflow->`largest_increase` (evacuation surge). RdBu, symmetric, **shared per column
across both storms** (red = drop, blue = surge). Verbatim reuse of `notebook/figure_flow_maps.ipynb`
repointed to the **100-mi** local results.

- Helene: cluster-level (101 clusters -> spread onto member counties via `county_cluster_assignments.csv`).
- Milton: county-level (`unit_id` is GEOID, 34 counties).
- Geometry/tracks from sibling `hurricane_oct/data/` (TIGER 2023, EPSG:5070). Requires the `geo` env.

Output -> `results/npj_100mi/figure5_flow_maps.{png,pdf}`.

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import LineString

ROOT = '/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category'
HO   = '/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/hurricane_oct'
COUNTY_SHP = f'{HO}/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp'
TRACK_SHP  = {'Helene': f'{HO}/data/storm_track/helene_storm_track.shp',
              'Milton': f'{HO}/data/storm_track/milton_storm_track.shp'}
LL  = f'{ROOT}/results/local_level'
OUT = f'{ROOT}/results/npj_100mi'

# 100-mi local-level analysis (vs 50-mi canonical in the original figure_flow_maps)
HURR = {'Helene': dict(dir='helene_100mi', level='cluster', landfall='2024-09-26'),
        'Milton': dict(dir='milton_100mi', level='county',  landfall='2024-10-09')}
# Column order: outflow (evacuation surge) -> within -> inflow
FLOWS = [('outflow','largest_increase','Outflow: largest increase (%)'),
         ('within','largest_drop','Within: largest drop (%)'),
         ('inflow','largest_drop','Inflow: largest drop (%)')]
FIG_W, FIG_H = 7.2, 5.4
OUTNAME = 'figure5_flow_maps'
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':7,'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':300})

In [2]:
counties = gpd.read_file(COUNTY_SHP)[['GEOID','STATEFP','geometry']].copy()
counties['GEOID'] = counties['GEOID'].astype(int)
counties = counties.to_crs(epsg=5070)
states = counties.dissolve(by='STATEFP')[['geometry']]

def load_track(p):
    t = gpd.read_file(p)
    line = LineString(t.geometry.tolist()) if (t.geom_type=='Point').all() else t.unary_union
    return gpd.GeoDataFrame(geometry=[line], crs=t.crs).to_crs(epsg=5070)
tracks = {h: load_track(p) for h,p in TRACK_SHP.items()}
print('counties', len(counties), '| tracks', list(tracks))

counties 3235 | tracks ['Helene', 'Milton']


In [3]:
def build_layer(hur, flow, col):
    cfg = HURR[hur]
    m = pd.read_csv(f"{LL}/{cfg['dir']}/metrics_{flow}.csv")[['unit_id', col]]
    m.columns = ['unit_id','value']
    if cfg['level']=='county':
        m = m.rename(columns={'unit_id':'GEOID'}); m['GEOID']=m['GEOID'].astype(int)
        return counties.merge(m[['GEOID','value']], on='GEOID', how='right')
    ca = pd.read_csv(f"{LL}/{cfg['dir']}/county_cluster_assignments.csv")[['GEOID','cluster']]
    ca['GEOID']=ca['GEOID'].astype(int)
    d = ca.merge(m.rename(columns={'unit_id':'cluster'}), on='cluster', how='left')
    # keep `cluster` so we can dissolve member counties into cluster outlines when plotting
    return counties.merge(d[['GEOID','cluster','value']], on='GEOID', how='right')

layers={}
for h in HURR:
    for flow,col,_ in FLOWS:
        g=build_layer(h,flow,col); layers[(h,flow)]=g
        print(f"{h:7s} {flow:8s} n={len(g):3d} valid={g['value'].notna().sum():3d} range=[{g['value'].min():6.1f},{g['value'].max():6.1f}]")
norms={}
for flow,col,_ in FLOWS:
    vals=pd.concat([layers[(h,flow)]['value'] for h in HURR]).dropna()
    # Robust symmetric cap (98th pct of |value|) so degenerate-baseline outliers
    # (e.g. Helene outflow +12595%% from a near-zero baseline cluster) don't crush the scale.
    vlim=float(np.nanpercentile(np.abs(vals),98)); true_max=float(np.nanmax(np.abs(vals)))
    norms[flow]=TwoSlopeNorm(vmin=-vlim,vcenter=0.0,vmax=vlim)
    print(f'{flow:8s} color cap +/-{vlim:.0f}%% (true max |val|={true_max:.0f}%%; beyond-cap clipped to extreme)')
CMAP='RdBu'

Helene  outflow  n=487 valid=487 range=[ -36.7,12595.4]
Helene  within   n=487 valid=487 range=[ -46.2,  -3.5]
Helene  inflow   n=487 valid=487 range=[ -51.9,  29.2]
Milton  outflow  n= 34 valid= 34 range=[ -55.9, 175.8]
Milton  within   n= 34 valid= 34 range=[ -46.3,  -9.4]
Milton  inflow   n= 34 valid= 34 range=[ -68.2,  -9.4]
outflow  color cap +/-145%% (true max |val|=12595%%; beyond-cap clipped to extreme)
within   color cap +/-35%% (true max |val|=46%%; beyond-cap clipped to extreme)
inflow   color cap +/-46%% (true max |val|=68%%; beyond-cap clipped to extreme)


In [4]:
fig, axes = plt.subplots(2, 3, figsize=(FIG_W, FIG_H), constrained_layout=True)
panel = iter('abcdef')
for r,h in enumerate(HURR):
    is_cluster = HURR[h]['level']=='cluster'
    # cluster panels: draw faint county sub-divisions, then dark cluster outlines on top.
    # county panels (Milton): the county IS the unit, so keep the normal edge as the outline.
    edge_c, edge_w = ('#cfcfcf', 0.1) if is_cluster else ('#666', 0.15)
    for c,(flow,col,_) in enumerate(FLOWS):
        ax=axes[r,c]; g=layers[(h,flow)]
        minx,miny,maxx,maxy=g.total_bounds; pad=0.05*max(maxx-minx,maxy-miny)
        x0,y0,x1,y1=minx-pad,miny-pad,maxx+pad,maxy+pad
        counties.cx[x0:x1,y0:y1].plot(ax=ax, facecolor='#f2f2f2', edgecolor='white', linewidth=0.25)
        states.cx[x0:x1,y0:y1].boundary.plot(ax=ax, color='#9a9a9a', linewidth=0.4)
        g.plot(ax=ax, column='value', cmap=CMAP, norm=norms[flow], edgecolor=edge_c, linewidth=edge_w,
               missing_kwds={'color':'#ddd','edgecolor':'#999','linewidth':0.15})
        if is_cluster:
            g.dissolve(by='cluster').boundary.plot(ax=ax, color='#222', linewidth=0.45)
        tracks[h].plot(ax=ax, color='black', linewidth=1.1)
        ax.set_xlim(x0,x1); ax.set_ylim(y0,y1); ax.set_aspect('equal'); ax.set_axis_off()
        ax.text(0.03,0.97,next(panel),transform=ax.transAxes,va='top',ha='left',fontweight='bold',fontsize=10)
        if r==0: ax.set_title(flow.capitalize(), fontweight='bold', fontsize=9, pad=3)
        if c==0: ax.text(-0.05,0.5,f"{h}\n({HURR[h]['landfall']})",transform=ax.transAxes,
                         rotation=90,va='center',ha='center',fontweight='bold',fontsize=9)
for c,(flow,col,label) in enumerate(FLOWS):
    sm=mpl.cm.ScalarMappable(norm=norms[flow],cmap=CMAP); sm.set_array([])
    cb=fig.colorbar(sm, ax=axes[:,c], location='bottom', shrink=0.85, aspect=26, pad=0.01)
    cb.set_label(label, fontsize=7.5); cb.ax.tick_params(labelsize=6.5)
os.makedirs(OUT, exist_ok=True)
for ext in ('png','pdf'): fig.savefig(f'{OUT}/{OUTNAME}.{ext}', dpi=400, bbox_inches='tight')
print('saved ->', f'{OUT}/{OUTNAME}.png / .pdf'); plt.close(fig)

saved -> /Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category/results/npj_100mi/figure5_flow_maps.png / .pdf


## Companion - raw flow-extreme visit counts (Helene vs Milton)

Histograms of the **raw visit counts** (trips/day, NOT percent-change-from-baseline) at each unit's window
extreme: outflow -> peak (max raw outflow), within & inflow -> trough (min raw within / inflow). Source =
`y_true` in `baseline_{flow}_{unit}.csv` over the 42-day event window (2024-09-20..10-31 for Helene; the
equivalent post-landfall window for Milton). Helene = 101 clusters, Milton = 34 counties; y-axis = fraction
of that storm's units (n differs). Raw counts span 3-4 orders of magnitude so the **x-axis is log-scaled**.

**NOTE:** raw magnitudes largely track unit population/size - Helene's clusters aggregate several counties,
Milton's units are single counties - so this contrasts absolute trip *volume*, not storm *impact*. (Impact,
normalized for size, is the percent-change DV shown in the maps.) Output ->
`results/npj_100mi/figure5b_flow_histograms.{png,pdf}`.

In [5]:
import glob
# ---- Companion: RAW per-unit visit-count extremes over the event window, Helene vs Milton ----
# RAW trips/day (NOT percent change). For each unit, over the 42-day event window:
#   within -> trough (min raw within visits), inflow -> trough (min raw inflow visits),
#   outflow -> peak (max raw outflow visits). Source = y_true in baseline_{flow}_{unit}.csv.
# Counts span 3-4 orders of magnitude -> log x-axis; y = fraction of units (n differs: 101 vs 34).
STORM_COLOR = {'Helene':'#4C72B0', 'Milton':'#C44E52'}
AGG  = {'within':'min', 'inflow':'min', 'outflow':'max'}
RLAB = {'outflow':'Outflow: peak raw visits (trips/day)',
        'within' :'Within: trough raw visits (trips/day)',
        'inflow' :'Inflow: trough raw visits (trips/day)'}
NBINS = 20

def raw_extremes(hur, flow):
    vals = []
    for f in glob.glob(f"{LL}/{HURR[hur]['dir']}/baseline_{flow}_*.csv"):
        y = pd.read_csv(f)['y_true'].dropna()
        if len(y): vals.append(y.min() if AGG[flow]=='min' else y.max())
    return np.asarray(vals, dtype=float)

rawx = {h: {flow: raw_extremes(h, flow) for flow,_,_ in FLOWS} for h in HURR}
for h in HURR:
    for flow,_,_ in FLOWS:
        s = rawx[h][flow]
        print(f"{h:7s} {flow:8s} ({AGG[flow]}) n={len(s):3d} range=[{s.min():,.0f}, {s.max():,.0f}] median={np.median(s):,.0f}")

figh, haxes = plt.subplots(1, 3, figsize=(7.6, 2.7), constrained_layout=True)
for c,(flow,col,_) in enumerate(FLOWS):
    ax = haxes[c]
    pooled = np.concatenate([rawx[h][flow] for h in HURR])
    bins = np.logspace(np.log10(pooled.min()), np.log10(pooled.max()), NBINS+1)
    for h in HURR:
        s = rawx[h][flow]
        ax.hist(s, bins=bins, weights=np.ones_like(s)/len(s), histtype='stepfilled', alpha=0.45,
                color=STORM_COLOR[h], edgecolor=STORM_COLOR[h], linewidth=1.1, label=f"{h} (n={len(s)})")
    ax.set_xscale('log'); ax.set_xlabel(RLAB[flow], fontsize=7.5)
    if c==0: ax.set_ylabel('Fraction of units', fontsize=7.5)
    ax.tick_params(labelsize=6.5)
    for sp in ('top','right'): ax.spines[sp].set_visible(False)
    ax.legend(fontsize=6, frameon=False, loc='upper left')
    ax.text(0.02,0.99,chr(97+c),transform=ax.transAxes,va='top',ha='left',fontweight='bold',fontsize=10)
for ext in ('png','pdf'): figh.savefig(f'{OUT}/figure5b_flow_histograms.{ext}', dpi=400, bbox_inches='tight')
print('saved ->', f'{OUT}/figure5b_flow_histograms.png / .pdf'); plt.close(figh)

Helene  outflow  (max) n=101 range=[1,163, 2,745,468] median=77,142
Helene  within   (min) n=101 range=[423, 18,635,712] median=186,849
Helene  inflow   (min) n=101 range=[438, 3,649,503] median=14,493
Milton  outflow  (max) n= 34 range=[5,554, 2,208,370] median=299,294
Milton  within   (min) n= 34 range=[3,156, 8,229,198] median=692,012
Milton  inflow   (min) n= 34 range=[1,438, 865,998] median=50,596


saved -> /Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category/results/npj_100mi/figure5b_flow_histograms.png / .pdf
